In [3]:
!pip install transformers datasets evaluate seqeval -q

In [4]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate

In [6]:
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

# Choose task
TASK = "pos_tags"   # or "chunk_tags"

label_list = dataset["train"].features[TASK].feature.names
num_labels = len(label_list)

print("Labels:", label_list)

C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kodur\.cache\huggingface\hub\datasets--eriktks--conll2003. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 14041 examples [00:00, 195999.69 examples/s]
Generating validation spl

Labels: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


In [7]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [8]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example[TASK][word_idx])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized["labels"] = labels
    return tokenized

tokenized_datasets = dataset.map(tokenize_and_align_labels)

Map: 100%|████████████████████████████████████████████████████████████████| 3453/3453 [00:02<00:00, 1290.56 examples/s]


In [9]:
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 6905.00it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",   # ✅ fixed
    save_strategy="no",
    logging_steps=50
)

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [13]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(label_list[p_val])
                curr_labels.append(label_list[l_val])

        true_preds.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [16]:
print("Training started...")
trainer.train()

Training started...


C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.575827,0.494112,0.851629,0.849922,0.850775,0.899298
2,0.360704,0.393831,0.870751,0.867067,0.868906,0.911579
3,0.336164,0.377900,0.874023,0.871298,0.872658,0.914912


C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\kodur\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: . seems not 

TrainOutput(global_step=750, training_loss=0.7183569488525391, metrics={'train_runtime': 987.7821, 'train_samples_per_second': 6.074, 'train_steps_per_second': 0.759, 'total_flos': 62603109021648.0, 'train_loss': 0.7183569488525391, 'epoch': 3.0})

In [18]:
nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

text = "John works at Google in California"

result = nlp(text)

for r in result:
    print(f"{r['word']} → {r['entity_group']} ({r['score']:.2f})")

john → NNP (0.96)
works → VBZ (0.80)
at → IN (0.95)
google → NNP (0.96)
in → IN (0.97)
california → NNP (0.96)
